In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys

project_path = '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM'

sys.path.append(project_path)

In [3]:
import pandas as pd

df = pd.read_csv(f'{project_path}/data/BBC News Train.csv')
df.head()

,ArticleId,Text,Category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1490 entries, 0 to 1489
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ArticleId  1490 non-null   int64 
 1   Text       1490 non-null   object
 2   Category   1490 non-null   object
dtypes: int64(1), object(2)
memory usage: 35.1+ KB


In [5]:
from newsbot.preprocessing import clean_text, preprocess_text

In [6]:
df['clean_text'] = df['Text'].apply(clean_text)
df['clean_text'] = df['clean_text'].apply(preprocess_text)
df.head()

,ArticleId,Text,Category,clean_text
0,1833,worldcom ex-boss launches defence lawyers defe...,business,worldcom exboss launch defence lawyer defend w...
1,154,german business confidence slides german busin...,business,german business confidence slide german busine...
2,1101,bbc poll indicates economic gloom citizens in ...,business,bbc poll indicate economic gloom citizen major...
3,1976,lifestyle governs mobile choice faster bett...,tech,lifestyle govern mobile choice fast well funky...
4,917,enron bosses in $168m payout eighteen former e...,business,enron boss m payout eighteen enron director ag...


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8
    )

tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_text'])
feature_names = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
tfidf_df['category'] = df['Category'].values


print(tfidf_df.shape)
tfidf_df.head()


(1490, 5000)


,abandon,abbas,abc,ability,able,abolish,abroad,absence,absolute,absolutely,...,youngster,youth,yuan,yugansk,yuganskneftegas,yukos,yukos claim,zealand,zero,zone
0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.022822,0.0,0.0,0.0,0.0,0.0,...,0.04096,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
def get_top_tfidf_terms(tfidf_df, category, top_n=10):

    category_df = tfidf_df[tfidf_df['category'] == category]

    mean_scores = category_df.drop('category', axis=1).mean().sort_values(ascending=False)

    return mean_scores.head(top_n)

In [9]:
for category in df['Category'].unique():
    print(f"\n{category.upper()}:")
    print(get_top_tfidf_terms(tfidf_df, category, top_n=10))


BUSINESS:
bn         0.055202
firm       0.038106
company    0.036787
market     0.033650
bank       0.033217
growth     0.033077
year       0.032909
economy    0.031713
sale       0.030885
rise       0.029583
dtype: float64

TECH:
mobile        0.050692
phone         0.046041
people        0.045032
technology    0.040171
game          0.037163
user          0.037086
service       0.035780
software      0.035574
computer      0.032531
microsoft     0.030291
dtype: float64

POLITICS:
mr            0.081713
labour        0.063110
election      0.059199
blair         0.053573
party         0.052303
government    0.044837
tory          0.044740
minister      0.041600
brown         0.036344
mr blair      0.032822
dtype: float64

SPORT:
win         0.051851
game        0.045261
play        0.039010
england     0.038223
player      0.032989
match       0.030878
champion    0.028278
cup         0.027796
team        0.026141
chelsea     0.025754
dtype: float64

ENTERTAINMENT:
film     0.100167

In [12]:
%%writefile /content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/td_idf_analysis.py

def get_top_tfidf_terms(tfidf_df, category, top_n=10):

    category_df = tfidf_df[tfidf_df['category'] == category]

    mean_scores = category_df.drop('category', axis=1).mean().sort_values(ascending=False)

    return mean_scores.head(top_n)

Writing /content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/td_idf_analysis.py
